<a href="https://colab.research.google.com/github/brsctoo/TilapiaFreshAI/blob/main/notebooks/TilapiaFreshAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PIC Tilápia — pré-processamento D0–D10
Executar as células em ordem. Cada grupo SEMO/OA/OC/OT/OO será analisado separadamente.
O pipeline lê os ZIPs sem alterar os originais, cria inventário, candidatos a recorte,
pranchas de revisão e características RGB/HSV/Lab/GLCM. A exportação final inclui apenas
recortes aprovados em `revisoes.csv`. As tabelas preliminares permitem conferir o resultado.

As características são medidas nas imagens segmentadas dos DOCX (filé sobre fundo preto).
As fotos JPG são cruas, com pano, bandeja, papel e etiquetas. Só entram quando a chave não
tem versão segmentada e, nesse caso, exigem máscara manual.

Os identificadores acompanham os mesmos filés entre dias, conforme informado pelo pesquisador.
O vínculo entre filés do mesmo peixe é desconhecido. A chave de identidade completa do filé
precisa ser confirmada antes de produzir partições para modelos. `sample_key` combina
C, grupo e concentração como referência de aquisição, sem atribuir identidade biológica.

A execução com o acervo real e a aprovação visual dos recortes ficam registradas nos resultados.
Saídas antigas do notebook foram removidas para evitar confusão com a execução desta versão.

In [ ]:
from pathlib import Path
import hashlib
import io
import json
import os
import re
import shutil
import tempfile
import unicodedata
import zipfile

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageOps
from scipy import ndimage

# Em Colab, monta o Drive. Localmente, informe os caminhos dos dois ZIPs abaixo.
try:
    from google.colab import drive
except ImportError:
    drive = None
if drive is not None:
    drive.mount('/content/drive')

PASTA_PROJETO = Path('/content/drive/MyDrive/PIC Tilápia')
ZIP_JPG = None  # Exemplo local: Path('/dados/dataset_tilapias_compactado.zip')
ZIP_DOCX = None
PASTA_SAIDA = PASTA_PROJETO / 'Preprocessamento'
LIMIAR_FUNDO = 30
NIVEIS_GLCM = 32
DISTANCIAS_GLCM = (1, 2, 4)
# Preencher somente após confirmar quais campos identificam um filé entre dias.
# Opções: ['c'] ou ['c', 'grupo', 'concentracao']; também é possível usar mapa CSV.
CAMPOS_ID_FILE = None
MAPA_FILE = None  # CSV com sample_key e fillet_id, identificadores lidos como texto.
VERSAO_PIPELINE = '2.1'


def resolver_caminho(caminho):
    caminho = Path(caminho).absolute()
    if caminho.exists():
        return caminho
    # Aceita os acentos compostos/decompostos usados no nome da pasta no Drive.
    atual = Path(caminho.anchor)
    for parte in caminho.parts[1:]:
        opcoes = [p for p in atual.iterdir()
                  if unicodedata.normalize('NFC', p.name) == unicodedata.normalize('NFC', parte)]
        if len(opcoes) != 1:
            raise FileNotFoundError(f'Caminho ausente ou ambíguo: {caminho}')
        atual = opcoes[0]
    return atual


def resolver_zip(pasta, nome):
    encontrados = [p for p in pasta.iterdir() if p.is_file() and p.name in (nome, nome + '.zip')]
    if len(encontrados) != 1:
        raise FileNotFoundError(f'Esperado um ZIP {nome} em {pasta}, encontrados {len(encontrados)}')
    return encontrados[0]


def sha256_arquivo(caminho):
    h = hashlib.sha256()
    with open(caminho, 'rb') as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b''):
            h.update(bloco)
    return h.hexdigest()


def salvar_csv(df, caminho):
    caminho = Path(caminho)
    temporario = caminho.with_suffix(caminho.suffix + '.tmp')
    df.to_csv(temporario, index=False)
    os.replace(temporario, caminho)

## Inventário e nomes
A chave da imagem inclui C, dia, grupo e concentração. `-8` no campo final é tratado
como o erro de separador observado em `C2 - D1 - OA - -8_.docx`, convertido em 0,8%.
Cada transformação é registrada. Concentrações fora de 0, 0,4, 0,8 e 1 são rejeitadas.
Os candidatos DOCX são escolhidos por área em pixels e o número de mídias é registrado.
Arquivos que compartilham a chave são mantidos no inventário. A preferência determinística
é DOCX segmentado, depois JPG; a revisão pode selecionar outro candidato pela coluna `usar`.

In [ ]:
def interpretar_nome(nome):
    original = Path(nome).name
    texto = original.upper()
    texto = re.sub(r'(C\d+\s*-\s*D\d+)\s+\d*\s*([A-Z]{2,4})\s*-', r'\1 - \2 -', texto)
    padrao = r'C(\d+)\s*-\s*D(\d+)\s*-\s*([A-Z0-9]+)\s*-\s*-?\s*([0-9O.,]+)_'
    match = re.fullmatch(padrao + r'\.(?:JPG|JPEG|PNG|DOCX)', texto)
    if match is None:
        raise ValueError('nome_fora_do_padrao')
    c, dia, grupo, token = match.groups()
    grupo = {'00': 'OO', '0T': 'OT', '0A': 'OA', '0C': 'OC'}.get(grupo, grupo)
    token = token.replace('O', '0').replace(',', '.')
    concentracoes = {'0': '0', '04': '04', '4': '04', '0.4': '04',
                     '08': '08', '8': '08', '0.8': '08', '1': '1', '01': '1'}
    if token not in concentracoes:
        raise ValueError('concentracao_desconhecida')
    conc = concentracoes[token]
    if grupo not in {'SEMO', 'OA', 'OC', 'OT', 'OO'} or not 0 <= int(dia) <= 10:
        raise ValueError('grupo_ou_dia_invalido')
    if (grupo == 'SEMO') != (conc == '0'):
        raise ValueError('grupo_concentracao_incompativeis')
    c, dia = str(int(c)), int(dia)
    chave = f'C{c}_D{dia}_{grupo}_{conc}'
    canonico = f'C{c} - D{dia} - {grupo} - {conc}_'
    return dict(c=c, dia=dia, grupo=grupo, concentracao=conc,
                concentracao_pct={'0': 0.0, '04': 0.4, '08': 0.8, '1': 1.0}[conc],
                image_key=chave, sample_key=f'C{c}_{grupo}_{conc}',
                nome_canonico=canonico,
                nome_corrigido=Path(original).stem != canonico)


def abrir_rgb(dados):
    with Image.open(io.BytesIO(dados)) as img:
        return ImageOps.exif_transpose(img).convert('RGB')


def imagem_docx(dados):
    candidatas = []
    with zipfile.ZipFile(io.BytesIO(dados)) as doc:
        for nome in sorted(doc.namelist()):
            if nome.startswith('word/media/') and Path(nome).suffix.lower() in {'.png', '.jpg', '.jpeg'}:
                conteudo = doc.read(nome)
                # Só o cabeçalho é lido aqui; apenas a maior mídia é decodificada.
                with Image.open(io.BytesIO(conteudo)) as img:
                    candidatas.append((img.width * img.height, nome, conteudo))
    if not candidatas:
        raise ValueError('docx_sem_imagem')
    _, nome, conteudo = max(candidatas, key=lambda x: (x[0], x[1]))
    return abrir_rgb(conteudo), nome, len(candidatas)


def construir_inventario(arquivos_zip, destino):
    destino = Path(destino)
    (destino / 'imagens').mkdir(parents=True, exist_ok=True)
    linhas = []
    for origem, arquivo in arquivos_zip.items():
        with zipfile.ZipFile(arquivo) as z:
            ocorrencias = {}
            for info in sorted(z.infolist(), key=lambda x: x.filename):
                if info.is_dir():
                    continue
                nome = info.filename
                # Inclui auxiliares no inventário, sem contá-los como imagens.
                ocorrencia = ocorrencias.get(nome, 0)
                ocorrencias[nome] = ocorrencia + 1
                uid = hashlib.sha256((origem + ':' + nome + ':' + str(ocorrencia)).encode()).hexdigest()[:24]
                row = dict(candidate_id=uid, origem=origem, arquivo_original=nome,
                           zip_original=Path(arquivo).name, ocorrencia_zip=ocorrencia, status='auxiliar', erro='')
                if Path(nome).suffix.lower() not in {'.jpg', '.jpeg', '.png', '.docx'}:
                    linhas.append(row)
                    continue
                try:
                    row.update(interpretar_nome(nome))
                    dados = z.read(info)
                    if nome.lower().endswith('.docx'):
                        img, midia, quantidade = imagem_docx(dados)
                    else:
                        img, midia, quantidade = abrir_rgb(dados), '', 1
                    pixels = np.asarray(img)
                    digest = hashlib.sha256(pixels.tobytes() + str(pixels.shape).encode()).hexdigest()
                    relativo = f'imagens/{uid}.png'
                    img.save(destino / relativo)
                    row.update(status='valido', largura=img.width, altura=img.height,
                               midia_docx=midia, numero_midias=quantidade, pixel_sha256=digest,
                               imagem=relativo, source_sha256=hashlib.sha256(dados).hexdigest())
                except Exception as exc:
                    row.update(status='rejeitado', erro=f'{type(exc).__name__}: {exc}')
                linhas.append(row)
    df = pd.DataFrame(linhas)
    validos = df[df.status == 'valido'].copy()
    if validos.empty:
        raise ValueError('Nenhuma imagem válida nos ZIPs; confira inventário e nomes.')
    # DOCX trazem o filé já segmentado sobre fundo preto. JPG são as fotos cruas,
    # com pano, bandeja, papel e etiquetas, e só entram quando não há versão segmentada.
    validos['prioridade'] = validos.origem.map({'docx': 0, 'jpg': 1})
    preferidos = validos.sort_values(['image_key', 'prioridade', 'arquivo_original']).drop_duplicates('image_key')
    df['preferido'] = df.candidate_id.isin(preferidos.candidate_id)
    df['duplicatas_chave'] = df.get('image_key', pd.Series(dtype=str)).map(validos.image_key.value_counts()).fillna(0).astype(int)
    salvar_csv(df, destino / 'inventario.csv')
    return df

## Recortes e aprovação
Nas imagens segmentadas (DOCX), o limiar de fundo gera uma proposta de máscara usando o
maior componente conectado. A proposta pode excluir partes do filé ou incluir restos do
fundo. Fotos JPG não segmentadas não recebem proposta: sem `mascara_manual`, entram em
`falhas_roi.csv`. Revise as pranchas e cada imagem necessária antes de aprovar. `revisoes.csv` persiste fora da pasta reconstruída.

Para cada candidato escolhido, preencha `usar=sim`, `aprovado=sim` e `observacao`.
Valores aceitos: `usar` sim/nao; `aprovado` sim/nao/pendente (maiúsculas e acentos são
ignorados). Outro valor interrompe a execução com a lista das linhas a corrigir.
As colunas x0, y0, x1, y1 limitam a área do recorte em coordenadas da imagem original
(limite final exclusivo); a proposta automática escolhe o maior componente dentro dessa
janela. Com coordenadas vazias, usa-se a imagem inteira.
Uma máscara manual PNG em `mascara_manual` deve ter o tamanho da imagem original,
com pixels do filé brancos e fundo preto. Ela prevalece sobre a proposta automática.
Um caminho relativo é lido a partir da pasta de saída.
Escolha apenas um candidato por `image_key`. Arquivos com o mesmo conteúdo sob chaves
diferentes precisam ser resolvidos escolhendo qual imagem usar. Após editar o CSV,
execute novamente as células a partir de “Executar pipeline”.

In [ ]:
def propor_mascara(rgb, janela=None):
    # Limite fixo e registrado, pensado para as imagens segmentadas (fundo preto).
    foreground = rgb.mean(axis=2) > LIMIAR_FUNDO
    if janela is not None:
        # A janela da revisão limita a busca antes de escolher o maior componente.
        foreground &= janela
    labels, quantidade = ndimage.label(foreground)
    if quantidade == 0:
        raise ValueError('imagem_sem_primeiro_plano')
    tamanhos = np.bincount(labels.ravel())
    tamanhos[0] = 0
    return ndimage.binary_fill_holes(labels == tamanhos.argmax())


def aplicar_recorte(rgb, revisao, origem, pasta_saida):
    janela = None
    coords = [str(revisao.get(k, '')).strip() for k in ('x0', 'y0', 'x1', 'y1')]
    if any(coords):
        if not all(coords):
            raise ValueError('bbox_incompleta')
        x0, y0, x1, y1 = map(int, coords)
        h, w = rgb.shape[:2]
        if not (0 <= x0 < x1 <= w and 0 <= y0 < y1 <= h):
            raise ValueError('bbox_fora_da_imagem')
        janela = np.zeros((h, w), dtype=bool)
        janela[y0:y1, x0:x1] = True
    manual = str(revisao.get('mascara_manual', '')).strip()
    if manual:
        caminho = Path(manual)
        if not caminho.is_absolute():
            caminho = Path(pasta_saida) / caminho
        with Image.open(caminho) as img:
            mascara = np.asarray(img.convert('L')) > 0
        if mascara.shape != rgb.shape[:2]:
            raise ValueError('mascara_manual_dimensao_incorreta')
        if janela is not None:
            mascara &= janela
    elif origem != 'docx':
        # Na foto crua o limiar seleciona pano, bandeja e papel junto com o filé.
        raise ValueError('foto_nao_segmentada_requer_mascara_manual')
    else:
        mascara = propor_mascara(rgb, janela)
    yy, xx = np.nonzero(mascara)
    if len(xx) < 64:
        raise ValueError('roi_menor_que_64_pixels')
    bbox = (int(xx.min()), int(yy.min()), int(xx.max() + 1), int(yy.max() + 1))
    return mascara, bbox


COLUNAS_REVISAO = ['usar', 'aprovado', 'x0', 'y0', 'x1', 'y1', 'mascara_manual', 'observacao', 'roi_sha256']
OPCOES_REVISAO = {'usar': {'sim', 'nao'}, 'aprovado': {'sim', 'nao', 'pendente'}}


def normalizar_opcao(valor):
    # Planilhas costumam capitalizar ou acentuar: 'Sim', ' SIM', 'Não'.
    texto = unicodedata.normalize('NFKD', str(valor)).encode('ascii', 'ignore').decode()
    return texto.strip().lower()


def atualizar_revisoes(inventario, caminho):
    """Combina as revisões gravadas com os candidatos desta execução. Não grava o arquivo."""
    validos = inventario[inventario.status == 'valido']
    novos = validos[['candidate_id', 'image_key', 'arquivo_original', 'pixel_sha256']].copy()
    novos['usar'] = np.where(validos.preferido, 'sim', 'nao')
    novos['aprovado'] = 'pendente'
    for col in COLUNAS_REVISAO[2:]:
        novos[col] = ''
    novos['ativo'] = 'sim'
    if caminho.exists():
        anterior = pd.read_csv(caminho, dtype=str, keep_default_na=False)
        if anterior.candidate_id.duplicated().any():
            raise ValueError('candidate_id repetido em revisoes.csv')
        for col in ['image_key', 'arquivo_original', 'pixel_sha256'] + COLUNAS_REVISAO:
            if col not in anterior.columns:
                anterior[col] = ''
        for col, opcoes in OPCOES_REVISAO.items():
            anterior[col] = anterior[col].map(normalizar_opcao)
            invalidos = anterior.loc[~anterior[col].isin(opcoes), 'candidate_id'].tolist()
            if invalidos:
                raise ValueError(f'revisoes.csv: valores de {col} fora de {sorted(opcoes)} '
                                 f'em {len(invalidos)} linha(s), por exemplo {invalidos[:5]}')
        anterior = anterior.set_index('candidate_id')
        herdados = set()
        for i, row in novos.iterrows():
            uid = row.candidate_id
            # Uma mudança nos pixels invalida a aprovação anterior.
            if uid in anterior.index and anterior.at[uid, 'pixel_sha256'] == row.pixel_sha256:
                for col in COLUNAS_REVISAO:
                    novos.at[i, col] = anterior.at[uid, col]
                herdados.add(uid)
        # Um candidato novo não disputa a chave com uma escolha já feita na revisão.
        escolhidas = set(novos.loc[novos.candidate_id.isin(herdados) & novos.usar.eq('sim'), 'image_key'])
        novos.loc[~novos.candidate_id.isin(herdados) & novos.image_key.isin(escolhidas), 'usar'] = 'nao'
        # Revisões de candidatos ausentes nesta execução ficam guardadas como inativas.
        ausentes = anterior.loc[~anterior.index.isin(novos.candidate_id)].reset_index()
        if not ausentes.empty:
            ausentes['ativo'] = 'nao'
            novos = pd.concat([novos, ausentes[novos.columns]], ignore_index=True)
    return novos.set_index('candidate_id')


def espacos_cor(rgb):
    rgb = rgb.astype(np.float64) / 255
    mx, mn = rgb.max(axis=2), rgb.min(axis=2)
    delta = mx - mn
    hue = np.zeros_like(mx)
    nonzero = delta > 0
    canal = rgb.argmax(axis=2)
    formulas = [(rgb[..., 1] - rgb[..., 2]),
                (rgb[..., 2] - rgb[..., 0]) + 2 * delta,
                (rgb[..., 0] - rgb[..., 1]) + 4 * delta]
    for i in range(3):
        m = nonzero & (canal == i)
        hue[m] = (formulas[i][m] / delta[m]) % 6
    hsv = np.stack([hue / 6, np.divide(delta, mx, out=np.zeros_like(mx), where=mx > 0), mx], axis=2)
    linear = np.where(rgb <= .04045, rgb / 12.92, ((rgb + .055) / 1.055) ** 2.4)
    xyz = linear @ np.array([[.4124564, .3575761, .1804375],
                             [.2126729, .7151522, .0721750],
                             [.0193339, .1191920, .9503041]]).T
    xyz /= np.array([.95047, 1., 1.08883])
    d = 6 / 29
    f = np.where(xyz > d ** 3, np.cbrt(xyz), xyz / (3 * d ** 2) + 4 / 29)
    lab = np.stack([116 * f[..., 1] - 16, 500 * (f[..., 0] - f[..., 1]),
                    200 * (f[..., 1] - f[..., 2])], axis=2)
    return {'rgb': rgb * 255, 'hsv': hsv, 'lab': lab}


def extrair_features(rgb, mascara):
    features = {}
    cores = espacos_cor(rgb)
    for espaco, imagem in cores.items():
        for canal, nome in enumerate(espaco):
            if espaco == 'hsv' and nome == 'h':
                continue  # Matiz é circular; estatísticas lineares não se aplicam.
            values = imagem[..., canal][mascara]
            prefixo = f'{espaco}_{nome}'
            features.update({prefixo + '_media': float(values.mean()),
                             prefixo + '_dp': float(values.std()),
                             prefixo + '_p10': float(np.percentile(values, 10)),
                             prefixo + '_p50': float(np.percentile(values, 50)),
                             prefixo + '_p90': float(np.percentile(values, 90))})
    # Matiz é circular (o vermelho fica em 0 e 1): média angular e concentração,
    # só nos pixels cromáticos, pois o matiz de cinza e branco é indefinido.
    hsv = cores['hsv']
    cromatico = mascara & (hsv[..., 1] > 0)
    hue = hsv[..., 0][cromatico] * 2 * np.pi
    if hue.size:
        seno, cosseno = np.sin(hue).mean(), np.cos(hue).mean()
        features['hue_media_circular'] = float(np.arctan2(seno, cosseno) % (2*np.pi) / (2*np.pi))
        features['hue_resultante'] = float(np.hypot(seno, cosseno))
    else:
        features['hue_media_circular'] = float('nan')
        features['hue_resultante'] = 0.0
    features['hue_fracao_cromatica'] = float(cromatico.sum() / mascara.sum())
    gray = np.rint(rgb @ np.array([.299, .587, .114])).astype(np.uint8)
    quant = np.minimum(gray.astype(int) * NIVEIS_GLCM // 256, NIVEIS_GLCM - 1)
    ii, jj = np.indices((NIVEIS_GLCM, NIVEIS_GLCM))
    h, w = mascara.shape
    # Pares somente quando os dois pixels pertencem à ROI, sem fundo artificial.
    for dist in DISTANCIAS_GLCM:
        medidas = []
        for dy, dx in [(0, dist), (dist, 0), (dist, dist), (dist, -dist)]:
            ya, yb = slice(0, h-dy), slice(dy, h)
            xa, xb = (slice(0, w-dx), slice(dx, w)) if dx >= 0 else (slice(-dx, w), slice(0, w+dx))
            if h <= dy or w <= abs(dx):
                raise ValueError('roi_pequena_para_distancia_glcm')
            valid = mascara[ya, xa] & mascara[yb, xb]
            if not valid.any():
                raise ValueError('roi_sem_pares_glcm')
            pares = quant[ya, xa][valid] * NIVEIS_GLCM + quant[yb, xb][valid]
            counts = np.bincount(pares, minlength=NIVEIS_GLCM**2).reshape(NIVEIS_GLCM, NIVEIS_GLCM)
            p = (counts + counts.T).astype(float)
            p /= p.sum()
            nz = p[p > 0]
            medidas.append([np.sum(p * (ii-jj)**2), np.sum(p / (1 + (ii-jj)**2)),
                            np.sqrt(np.sum(p*p)), -np.sum(nz*np.log2(nz))])
        for nome, valor in zip(['contraste', 'homogeneidade', 'energia', 'entropia'], np.mean(medidas, axis=0)):
            features[f'glcm_d{dist}_{nome}'] = float(valor)
    return features

## Executar pipeline
As propostas e as características preliminares ficam em `atual/`. As revisões persistem
em `revisoes.csv`, que só é atualizado quando a execução termina; linhas de candidatos que
sumiram dos ZIPs ficam guardadas com `ativo=nao`. As pranchas mostram original e recorte mascarado com o identificador.
`features_final.csv` contém apenas candidatos selecionados, aprovados e sem conflitos.
`resumo.json` informa pendências, contagens e hashes dos insumos. A execução reconstrói
os resultados em uma pasta temporária antes de substituir `atual/`.

In [ ]:
def vincular_file(df):
    df = df.copy()
    df['fillet_id'] = ''
    df['fish_id'] = ''
    if MAPA_FILE is not None:
        mapa = pd.read_csv(MAPA_FILE, dtype=str, keep_default_na=False)
        if not {'sample_key', 'fillet_id'} <= set(mapa.columns) or mapa.sample_key.duplicated().any():
            raise ValueError('Mapa deve ter sample_key única e fillet_id.')
        df['fillet_id'] = df.sample_key.map(mapa.set_index('sample_key').fillet_id).fillna('')
    elif CAMPOS_ID_FILE:
        if not set(CAMPOS_ID_FILE) <= {'c', 'grupo', 'concentracao'}:
            raise ValueError('Campos de identidade de filé inválidos.')
        df['fillet_id'] = df[CAMPOS_ID_FILE].astype(str).agg('|'.join, axis=1)
    return df


def gerar_pranchas(linhas, destino):
    linhas = sorted(linhas, key=lambda r: (r['grupo'], r['dia'], r['image_key'], r['candidate_id']))
    for inicio in range(0, len(linhas), 12):
        pagina = Image.new('RGB', (1000, 12 * 155), 'white')
        draw = ImageDraw.Draw(pagina)
        for pos, row in enumerate(linhas[inicio:inicio+12]):
            y = pos * 155
            for x, campo in [(0, 'imagem'), (500, 'preview')]:
                with Image.open(destino / row[campo]) as img:
                    img = img.convert('RGB')
                    img.thumbnail((490, 115))
                    pagina.paste(img, (x, y + 35))
            texto = f"{row['candidate_id']} {row['image_key']} {row['origem']} {row['aprovado']}"
            draw.text((5, y+3), texto, fill='black')
        pagina.save(destino / f'prancha_{inicio//12+1:03d}.jpg')


def _executar_pipeline(arquivos_zip, pasta_saida):
    pasta_saida = Path(pasta_saida)
    pasta_saida.mkdir(parents=True, exist_ok=True)
    staging = Path(tempfile.mkdtemp(prefix='execucao_', dir=pasta_saida))
    try:
        inventario = construir_inventario(arquivos_zip, staging)
        # revisoes.csv só é substituído depois que atual/ recebe as pranchas desta execução,
        # para que nunca registre recortes diferentes dos que estão à vista.
        revisoes = atualizar_revisoes(inventario, pasta_saida / 'revisoes.csv')
        validos = inventario[inventario.status == 'valido'].copy()
        validos = vincular_file(validos)
        selecionados = validos[validos.candidate_id.map(revisoes.usar).eq('sim')]
        conflitos_chave = set(selecionados.loc[selecionados.image_key.duplicated(False), 'image_key'])
        conflitos_hash = set(selecionados.loc[selecionados.pixel_sha256.duplicated(False), 'pixel_sha256'])
        linhas, falhas, pranchas = [], [], []
        (staging / 'recortes').mkdir()
        for _, row in validos.iterrows():
            registro = row.to_dict()
            rev = revisoes.loc[row.candidate_id].to_dict()
            registro.update({k: rev[k] for k in ['usar', 'aprovado', 'observacao']})
            registro['conflito'] = row.image_key in conflitos_chave or row.pixel_sha256 in conflitos_hash
            try:
                with Image.open(staging / row.imagem) as img:
                    rgb = np.asarray(img.convert('RGB'))
                mascara, bbox = aplicar_recorte(rgb, rev, row.origem, pasta_saida)
                x0, y0, x1, y1 = bbox
                recorte = rgb[y0:y1, x0:x1]
                mask = mascara[y0:y1, x0:x1]
                preview = np.where(mask[..., None], recorte, 0).astype(np.uint8)
                relativo = f'recortes/{row.candidate_id}.png'
                Image.fromarray(preview).save(staging / relativo)
                mask_path = f'recortes/{row.candidate_id}_mask.png'
                Image.fromarray(mask.astype(np.uint8)*255).save(staging / mask_path)
                registro.update(x0=x0, y0=y0, x1=x1, y1=y1,
                                roi_pixels=int(mask.sum()), roi_fracao=float(mascara.mean()),
                                roi_sha256=hashlib.sha256(preview.tobytes()+mask.tobytes()).hexdigest(),
                                mascara_sha256=hashlib.sha256(mascara.tobytes()).hexdigest(),
                                preview=relativo, mascara=mask_path)
                if rev.get('roi_sha256', '') != registro['roi_sha256']:
                    registro['aprovado'] = 'pendente'
                    revisoes.at[row.candidate_id, 'aprovado'] = 'pendente'
                revisoes.at[row.candidate_id, 'roi_sha256'] = registro['roi_sha256']
                registro.update(extrair_features(recorte, mask))
                linhas.append(registro)
                pranchas.append(registro)
            except Exception as exc:
                registro['erro'] = f'{type(exc).__name__}: {exc}'
                falhas.append(registro)
        salvar_csv(revisoes.reset_index(), staging / 'revisoes.csv')
        preliminar = pd.DataFrame(linhas)
        # Sem nenhuma ROI (por exemplo, só fotos JPG ainda sem máscara manual), a execução
        # publica revisoes.csv e falhas_roi.csv para que as máscaras possam ser indicadas.
        if preliminar.empty:
            final = preliminar
        else:
            final = preliminar[preliminar.usar.eq('sim') & preliminar.aprovado.eq('sim') & ~preliminar.conflito].copy()
        salvar_csv(preliminar, staging / 'features_preliminares.csv')
        salvar_csv(final, staging / 'features_final.csv')
        salvar_csv(pd.DataFrame(falhas), staging / 'falhas_roi.csv')
        # Contagem de imagens por chave antes de aprovar os recortes.
        unicos = validos[validos.preferido]
        # Fontes disponíveis por chave: 'docx' (recuperada só do segmentado), 'jpg' (precisa
        # de máscara manual) ou 'docx+jpg'.
        fontes = validos.groupby('image_key').origem.agg(lambda s: '+'.join(sorted(set(s))))
        contagens = unicos.groupby(['dia', 'grupo']).size().unstack(fill_value=0)
        contagens = contagens.reindex(index=range(11), columns=['SEMO', 'OA', 'OC', 'OT', 'OO'], fill_value=0)
        contagens.reset_index().to_csv(staging / 'contagens_dia_grupo.csv', index=False)
        validos.groupby('origem').agg(arquivos=('candidate_id', 'size'),
                                     largura_mediana=('largura', 'median'),
                                     altura_mediana=('altura', 'median')).to_csv(staging / 'qualidade_por_origem.csv')
        comparacao = validos.groupby('image_key').agg(candidatos=('candidate_id', 'size'),
                                                    conteudos_distintos=('pixel_sha256', 'nunique'))
        comparacao[comparacao.candidatos > 1].reset_index().to_csv(staging / 'comparacao_fontes.csv', index=False)
        gerar_pranchas(pranchas, staging)
        resumo = dict(versao=VERSAO_PIPELINE,
                      zip_sha256={k: sha256_arquivo(v) for k, v in arquivos_zip.items()},
                      parametros=dict(limiar_fundo=LIMIAR_FUNDO, niveis_glcm=NIVEIS_GLCM,
                                      distancias_glcm=DISTANCIAS_GLCM, campos_id_file=CAMPOS_ID_FILE,
                                      escala_hsv='H,S,V em 0..1', lab='sRGB D65',
                                      glcm='simetrica, media de 4 direcoes, distancias em pixels'),
                      inventario_status=inventario.status.value_counts().to_dict(),
                      chaves_unicas=len(unicos), candidatos_validos=len(validos),
                      recuperadas_docx=int((fontes == 'docx').sum()),
                      chaves_por_fonte={k: int(v) for k, v in sorted(fontes.value_counts().items())},
                      conflitos_chave=sorted(conflitos_chave), conflitos_hash=sorted(conflitos_hash),
                      roi_falhas=len(falhas), features_preliminares=len(preliminar),
                      features_finais=len(final),
                      selecionados_pendentes=int((selecionados.candidate_id.map(revisoes.aprovado) != 'sim').sum()),
                      exportacao_completa=bool(len(selecionados) and len(final) == len(selecionados)),
                      identidade_file_completa=bool(len(final) and final.fillet_id.ne('').all()),
                      identidade_peixe_disponivel=False,
                      limitacao='Vinculo entre files do mesmo peixe desconhecido; risco residual entre particoes.',
                      revisoes_sha256=sha256_arquivo(staging / 'revisoes.csv'))
        (staging / 'resumo.json').write_text(json.dumps(resumo, ensure_ascii=False, indent=2))
        atual, anterior = pasta_saida / 'atual', pasta_saida / 'anterior'
        if anterior.exists():
            shutil.rmtree(anterior)
        if atual.exists():
            atual.rename(anterior)
        try:
            staging.rename(atual)
        except Exception:
            if anterior.exists() and not atual.exists():
                anterior.rename(atual)
            raise
        # Move (não copia) para que só exista um revisoes.csv editável.
        os.replace(atual / 'revisoes.csv', pasta_saida / 'revisoes.csv')
        return resumo
    except Exception:
        # Preserva a execução anterior completa e remove apenas o temporário desta execução.
        if staging.exists():
            shutil.rmtree(staging)
        raise

In [ ]:
def executar_pipeline(arquivos_zip, pasta_saida):
    pasta_saida = Path(pasta_saida)
    pasta_saida.mkdir(parents=True, exist_ok=True)
    lock = pasta_saida / '.execucao_em_andamento'
    try:
        lock.mkdir()
    except FileExistsError:
        raise RuntimeError(f'Outra execução usa esta saída. Se foi interrompida, remova {lock} antes de tentar novamente.')
    try:
        return _executar_pipeline(arquivos_zip, pasta_saida)
    finally:
        lock.rmdir()

## Verificações de regressão
Executadas sem acesso ao Drive. Conferem o parser, conversões de cor conhecidas e
GLCM em imagem uniforme. A validação com os ZIPs, a contagem real de imagens e a
aprovação visual dos recortes são etapas registradas em `resumo.json` após a execução.

In [ ]:
def _selftest():
    casos = {'C2 - D1 - OA - -8_.docx': ('OA', '08'),
             'C2 - D5 0 OC - 1_.docx': ('OC', '1'),
             'C2 - D5 - 00 - 1_.png': ('OO', '1'),
             'C2 - D6 - 0T - 08_.png': ('OT', '08'),
             'C1-D0-OT-0,4_.jpg': ('OT', '04'),
             'C2 - D4 - SEMO - O_.jpg': ('SEMO', '0')}
    for nome, esperado in casos.items():
        obtido = interpretar_nome(nome)
        assert (obtido['grupo'], obtido['concentracao']) == esperado, nome
    try:
        interpretar_nome('C1 - D11 - SEMO - 0_.jpg')
    except ValueError:
        pass
    else:
        raise AssertionError('Dia inválido aceito')
    branco = np.full((12, 12, 3), 255, dtype=np.uint8)
    cores = espacos_cor(branco)
    assert np.allclose(cores['lab'][0, 0], [100, 0, 0], atol=.001)
    features = extrair_features(branco, np.ones((12, 12), dtype=bool))
    assert features['glcm_d1_contraste'] == 0
    assert features['glcm_d1_homogeneidade'] == 1
    assert features['glcm_d1_energia'] == 1
    assert features['glcm_d1_entropia'] == 0
    print('selftest: ok')

_selftest()

In [ ]:
# Ajuste ZIP_JPG e ZIP_DOCX para executar fora do Colab.
if ZIP_JPG is None or ZIP_DOCX is None:
    PASTA_PROJETO = resolver_caminho(PASTA_PROJETO)
    pasta_dataset = PASTA_PROJETO / 'Dataset'
    ZIP_JPG = resolver_zip(pasta_dataset, 'dataset_tilapias_compactado') if ZIP_JPG is None else Path(ZIP_JPG)
    ZIP_DOCX = resolver_zip(pasta_dataset, 'dataset_tilapias_segmentado_compactado') if ZIP_DOCX is None else Path(ZIP_DOCX)

resumo = executar_pipeline({'jpg': ZIP_JPG, 'docx': ZIP_DOCX}, PASTA_SAIDA)
print(json.dumps(resumo, ensure_ascii=False, indent=2))
print('Revise as pranchas e preencha:', PASTA_SAIDA / 'revisoes.csv')
print('Resultados:', PASTA_SAIDA / 'atual')

## Partições por filé
Configure `CAMPOS_ID_FILE` ou `MAPA_FILE` após confirmar a chave completa.
A função abaixo monta uma partição por filé dentro de cada tratamento e exige
D0–D10 em ambos os lados. Poucos filés podem inviabilizar essa divisão, o que é informado
explicitamente. A escolha dos filés de teste deve preceder o treino. Normalização,
seleção de características e aumento de dados devem ser ajustados dentro do treino.
A identidade dos peixes continua desconhecida mesmo com a separação por filé.

In [ ]:
def preparar_particao(features, grupo, files_teste):
    dados = features[features.grupo == grupo].copy()
    if dados.empty or dados.fillet_id.fillna('').eq('').any():
        raise ValueError('Confirme os identificadores de filé antes de criar partições.')
    files_teste = set(files_teste)
    if not files_teste or not files_teste <= set(dados.fillet_id):
        raise ValueError('Informe filés de teste presentes neste grupo.')
    dados['particao'] = np.where(dados.fillet_id.isin(files_teste), 'teste', 'treino')
    for particao in ['treino', 'teste']:
        if set(dados.loc[dados.particao == particao, 'dia'].astype(int)) != set(range(11)):
            raise ValueError(f'{particao} precisa conter D0–D10. Confira o número de filés disponíveis.')
    ids_treino = set(dados.loc[dados.particao == 'treino', 'fillet_id'])
    ids_teste = set(dados.loc[dados.particao == 'teste', 'fillet_id'])
    assert not ids_treino.intersection(ids_teste)
    return dados